In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Load your data (Example: 1000 reviews)
df = pd.read_csv('amazon_reviews.csv')

In [2]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# 1. Initialize VADER
nltk.download('vader_lexicon')
sid = SentimentIntensityAnalyzer()

# 2. Calculate the AI Score FIRST (Step 2)
def get_vader_score(text):
    if pd.isna(text): return 0.0
    return sid.polarity_scores(str(text))['compound']

# Create the new column 'ai_sentiment_score'
df['ai_sentiment_score'] = df['Text'].apply(get_vader_score)

# 3. Updated Audit Logic (The Fix)
def design_audit_logic(row):
    text = str(row['Text']).lower()
    ai_score = row['ai_sentiment_score']  # AI confidence (-1 to 1)
    user_stars = row['Score']             # User rating (1 to 5)
    
    # FIX: Rule 1 - Confidence Threshold (Trigger if AI is "unsure")
    # AI is "unsure" when the score is close to 0 (neutral)
    if -0.2 <= ai_score <= 0.2:
        return True, "Low AI Confidence"

    # FIX: Rule 2 - Discrepancy (AI and Human disagree)
    # If AI says Positive (>0.5) but User gave 1 or 2 stars
    if (ai_score > 0.5 and user_stars <= 2) or (ai_score < -0.5 and user_stars >= 4):
        return True, "Sentiment Discrepancy"

    # Rule 3: Contrastive Language
    contrast_words = ['but', 'however', 'yet', 'although']
    if any(word in text for word in contrast_words):
        return True, "Contrastive Language"

    return False, "Safe"

# 4. Apply the updated logic
df[['audit_required', 'audit_reason']] = df.apply(
    lambda row: pd.Series(design_audit_logic(row)), axis=1
)

# 5. Save the Audit File
audit_file = df[df['audit_required'] == True]
audit_file.to_csv('ai_audit_queue.csv', index=False)

print(f"Audit file saved! Total items to review: {len(audit_file)}")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sony\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Audit file saved! Total items to review: 309692


In [3]:
# Create a sample of reviews that need human auditing
audit_queue = df[df['audit_required'] == True].head(5)

# Simulate the Human-in-the-Loop Correction
verified_entries = []

for index, row in audit_queue.iterrows():
    print(f"\nReview Text: {row['Text']}")
    print(f"AI Reason for Audit: {row['audit_reason']}")
    
    # In a real tool, this would be a UI button. Here, it's a manual input.
    user_correction = input("Enter Correct Sentiment (Positive/Negative/Neutral): ")
    
    verified_entries.append({
        'Id': row['Id'],
        'Original_Text': row['Text'],
        'AI_Label': row['audit_reason'], # Or your AI sentiment label
        'Human_Label': user_correction,
        'Status': 'Verified'
    })

# Save the 'Ground Truth' data
verified_df = pd.DataFrame(verified_entries)
verified_df.to_csv('peroptyx_ground_truth.csv', index=False)
print("\nSuccess: Verified dataset created for retraining!")


Review Text: If you are looking for the secret ingredient in Robitussin I believe I have found it.  I got this in addition to the Root Beer Extract I ordered (which was good) and made some cherry soda.  The flavor is very medicinal.
AI Reason for Audit: Low AI Confidence


Enter Correct Sentiment (Positive/Negative/Neutral):  Positive



Review Text: I don't know if it's the cactus or the tequila or just the unique combination of ingredients, but the flavour of this hot sauce makes it one of a kind!  We picked up a bottle once on a trip we were on and brought it back home with us and were totally blown away!  When we realized that we simply couldn't find it anywhere in our city we were bummed.<br /><br />Now, because of the magic of the internet, we have a case of the sauce and are ecstatic because of it.<br /><br />If you love hot sauce..I mean really love hot sauce, but don't want a sauce that tastelessly burns your throat, grab a bottle of Tequila Picante Gourmet de Inclan.  Just realize that once you taste it, you will never want to use any other sauce.<br /><br />Thank you for the personal, incredible service!
AI Reason for Audit: Contrastive Language


Enter Correct Sentiment (Positive/Negative/Neutral):  Positive



Review Text: One of my boys needed to lose some weight and the other didn't.  I put this food on the floor for the chubby guy, and the protein-rich, no by-product food up higher where only my skinny boy can jump.  The higher food sits going stale.  They both really go for this food.  And my chubby boy has been losing about an ounce a week.
AI Reason for Audit: Sentiment Discrepancy


...


Review Text: I love eating them and they are good for watching TV and looking at movies! It is not too sweet. I like to transfer them to a zip lock baggie so they stay fresh so I can take my time eating them.
AI Reason for Audit: Sentiment Discrepancy


Enter Correct Sentiment (Positive/Negative/Neutral):  Negative



Review Text: I love this candy.  After weight watchers I had to cut back but still have a craving for it.
AI Reason for Audit: Contrastive Language


Enter Correct Sentiment (Positive/Negative/Neutral):  Positive



Success: Verified dataset created for retraining!
